# 02 - Kinematic PCA per phase

**Purpose**: fit PCA separately on each experimental phase's
(subject, contacted) kinematic summary and compare loading patterns
across phases.

**Input**: ``data.AKDdf_agg_contact`` (aggregated kinematics by
contact_group), flattened.

**Caveat**: at small N each per-phase PCA has the matched-subject count as its row count x many features --
PC1 is well-estimated, higher PCs are noisy.


In [ ]:
# parameters
PHASES = ['Baseline', 'Post_Injury_1', 'Post_Injury_2-4', 'Post_Rehab_Test']
N_COMPONENTS = 3
TOP_N_FEATURES = 5  # Features to count as "important" per PC when pooling across phases
FIGSIZE_HEATMAP = (24, 10)


In [ ]:
import numpy as np                                                       # numerical arrays
import pandas as pd                                                      # dataframe library
import matplotlib.pyplot as plt                                          # plot interface
import seaborn as sns                                                    # statistical plotting (heatmaps here)

from endpoint_ck_analysis.config import CACHE_DIR, EXAMPLE_OUTPUT_DIR    # cache_dir for parquet writes; example_output_dir for saved PNGs
from endpoint_ck_analysis.data_loader import load_all                    # one-shot data loader
from endpoint_ck_analysis.helpers.dimreduce import run_pca_for_phase, align_signs_to_reference # per-phase PCA helper + cross-phase sign alignment
from endpoint_ck_analysis.helpers.plotting import plot_pca_for_phase, stamp_version            # per-phase scree+loadings plot helper + figure version footer

EXAMPLE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)                    # ensure output folder exists
data = load_all()                                                        # load (uses cache from 00_setup)
agg_flat = data.AKDdf_agg_contact_flat()                                 # flatten the multi-index AKDdf_agg_contact so we can filter on phase_group + contact_group as regular columns


## 1. Fit PCA per phase

``run_pca_for_phase`` filters to the phase + contacted reaches, scales,
runs PCA, and returns the fit plus its scores / loadings.


In [ ]:
pcas = {phase: run_pca_for_phase(agg_flat, phase, n_components=N_COMPONENTS) for phase in PHASES} # dict-comprehension: one PCA fit per phase, key = phase name, value = the helper's tuple return
for phase, (pca, _, eigen, _, _) in pcas.items():                                                  # iterate over each phase's result; the underscores discard the scores/loadings/X we don't need for this print
    print(f'{phase}: variance explained = {pca.explained_variance_ratio_.round(3)}')               # .round(3) limits to 3 decimal places for readability


**What you just saw.** The per-phase variance-explained ratios. Each
line is one phase, with the variance share captured by PC1, PC2,
PC3 within that phase.

**What different patterns mean.** Similar ratios across all four
phases = the dimensionality of how mice differ in reaching is stable
across the experiment; the same number of "main directions" exists
before injury, post-injury, and post-ABT. Very different ratios =
the kinematic state space changes shape with phase. *Example:* if
Baseline shows PC1=0.6 / PC2=0.2 / PC3=0.1 but Post_Injury_2-4
shows PC1=0.3 / PC2=0.3 / PC3=0.2, injury produced a wider spectrum
of compensatory reaching styles, with each mouse exploring its own
independent strategy.


## 2. Per-phase scree + loading figures

``plot_pca_for_phase`` emits one scree plot and a loading-bars figure
per phase. Saves PNGs for the gallery.


In [ ]:
for phase, (pca, _, eigen, loadings, _) in pcas.items():     # iterate phase by phase
    plot_pca_for_phase(pca, eigen, loadings, phase)          # helper renders scree + loading bars; uses plt.show() so each phase appears as its own pair of inline figures


**What these figures show.** For each phase, two figures: a scree plot
(variance per PC) and a loading bars panel (which kinematic features
drive each PC at that phase).

**Top loadings -- what they mean.** A feature with a long bar on
PC1 at a given phase is one along which mice differ a lot at that
phase. *Example:* if `peak_velocity_mm_per_sec` loads strongly on
PC1 at Post_Injury_1 but weakly at Baseline, injured mice differ
greatly in how fast they reach (some retain near-baseline speed,
others are dramatically slowed) while baseline mice all reach at
similar speeds.

**Low loadings -- what they mean.** A feature with a near-zero bar
doesn't separate mice at that phase along that PC. The feature is
either uniformly performed across mice (all reaching alike) or its
variance lives on a higher PC the loop didn't display. Either way,
it can't distinguish individuals on the dominant axis at that phase.

**Cross-phase comparison.** Compare the four scree plots side by
side. If PC1's variance share is much smaller post-injury than at
baseline, injury fragmented the cohort into more independent
reaching styles. Compare the four loading bars: a feature that
loads big on every phase is doing similar work throughout the
experiment; a feature that swings between phases changes its role
depending on injury state.


## 3. Align PC signs across phases

Per-phase PCA produces PCs with arbitrary sign. Aligning each
non-baseline phase's PCs to Baseline (flipping if Pearson r < 0) so
"positive PC1 loading" means the same thing across phases.


In [ ]:
loadings_baseline = pcas['Baseline'][3]                                                # index 3 = the loadings DataFrame in the helper's tuple (pca, scores, eigen, loadings, X)
loadings_by_phase = {'Baseline': loadings_baseline}                                    # baseline serves as the sign reference; other phases get aligned to it
for phase in PHASES[1:]:                                                                # PHASES[1:] = every phase except Baseline
    loadings_by_phase[phase] = align_signs_to_reference(pcas[phase][3], loadings_baseline) # flip a non-baseline phase's PC sign if it correlates negatively with baseline's same-numbered PC

loadings_concat = pd.concat(loadings_by_phase, axis=0)                                  # stack all phases vertically; result has a 2-level index (phase, PC) so we can pivot per-PC heatmaps later


## 4. Heatmap: per-PC loadings across phases

Each subplot is one PC. Rows are kinematic features, columns are phases.
Red = positive loading, blue = negative.


In [ ]:
fig, axes = plt.subplots(1, N_COMPONENTS, figsize=FIGSIZE_HEATMAP)         # one row of subplots, one per PC
for i, pc in enumerate([f'PC{k+1}' for k in range(N_COMPONENTS)]):          # iterate PC1, PC2, ... PC{N_COMPONENTS}
    pc_loadings = loadings_concat.xs(pc, level=1).T                         # .xs cross-section: pull just this PC across all phases; .T transposes so features become rows and phases become columns
    sns.heatmap(pc_loadings, cmap='RdBu_r', center=0, ax=axes[i],           # red-blue colormap centered at 0 so positive vs negative loadings are visually distinct
                cbar_kws={'label': 'Loading'})                              # colorbar legend label
    axes[i].set_title(f'{pc} loadings across phases')                       # subplot title
plt.tight_layout()                                                          # auto-adjust subplot spacing so titles/axes don't overlap
stamp_version(fig, label='02 loadings by phase')
plt.savefig(EXAMPLE_OUTPUT_DIR / '02_loadings_by_phase.png', dpi=150, bbox_inches='tight')
plt.show()


**What this figure shows.** One subplot per PC, with a row per
kinematic feature and a column per phase. Red = positive loading,
blue = negative loading, intensity = magnitude. Signs are aligned to
Baseline so a "red cell at Baseline" and a "red cell at Post_Rehab"
on the same row mean the feature is contributing in the same
direction at both phases.

**What different patterns mean.**
- A row that's consistently red (or blue) across all four columns =
  the feature plays the same role in the dominant variance axis at
  every phase. It's a stable kinematic axis.
- A row that flips color between columns = the feature's role
  depends on phase. *Example:* `peak_velocity_mm_per_sec` red at
  Baseline but blue at Post_Injury_2-4 would mean fast-reaching
  mice are at one end of PC1 at baseline but at the other end
  post-injury -- the feature changed its diagnostic meaning across
  experimental phases.
- A row that's pale (near-zero) at one or two phases but bold at
  others = the feature only contributes to PC structure at those
  specific phases. Useful when interpreting what each phase's PC1
  actually captures.

**Scientific reading.** Stable rows are kinematic axes the paper
can describe phase-invariantly. Flipping or swinging rows are
features whose interpretation depends on when in the experiment
you measure -- the paper either reports them per-phase or skips
them as ambiguous.


## 5. Identify important features (mean |loading| across phases)

Pool each PC's absolute loadings across phases; the top ``TOP_N_FEATURES``
per PC are the kinematic features most consistently structured by the
dominant subject-level variance. Write their union to the cache for
the PLS notebook.


In [ ]:
important_features = {}                                                  # PC name -> Series of top features by mean |loading|
for pc in [f'PC{k+1}' for k in range(N_COMPONENTS)]:                     # iterate over PC1, PC2, ... PC{N_COMPONENTS}
    pc_loadings = loadings_concat.xs(pc, level=1).T                       # features x phases for this PC (same pivot as the heatmap above)
    mean_abs = pc_loadings.abs().mean(axis=1)                             # collapse phases: each feature gets a single mean-absolute-loading score across phases
    top = mean_abs.nlargest(TOP_N_FEATURES)                               # keep only the TOP_N_FEATURES strongest contributors for this PC
    important_features[pc] = top
    print(f'\nTop {TOP_N_FEATURES} features on {pc} by mean |loading|:')
    print(top)

all_important = set()                                                    # union of important features across PCs
for pc, series in important_features.items():
    all_important.update(series.index)
print(f'\n{len(all_important)} unique important features total')

pd.Series(sorted(all_important), name='feature').to_frame().to_parquet(  # write the union to parquet for notebook 04 to consume; sorted for deterministic file content
    CACHE_DIR / 'important_features.parquet', index=False
)


**What you just saw.** Per-PC top features, then their union. The
union is what gets carried forward as the kinematic Y-block for PLS
in 04 -- only features whose role in the variance structure is
substantial on SOME PC participate in the coupling-to-connectivity
question. Features that never make any PC's top list contribute
mostly noise from this analysis's perspective.

**Reading the lists.** A feature appearing in PC1's top set across
multiple phases is one whose role in cohort variance is robust. A
feature appearing only on a single PC is a secondary contributor.
A feature appearing nowhere is dropped.


## 6. Heatmap filtered to important features


In [ ]:
fig, axes = plt.subplots(1, N_COMPONENTS, figsize=FIGSIZE_HEATMAP)         # one subplot per PC, same layout as the all-features heatmap
for i, pc in enumerate([f'PC{k+1}' for k in range(N_COMPONENTS)]):
    pc_loadings = loadings_concat.xs(pc, level=1).T                         # features x phases for this PC
    pc_top = pc_loadings.loc[pc_loadings.index.isin(all_important)]         # filter rows to only the important-feature set computed in the previous cell
    sns.heatmap(pc_top, cmap='RdBu_r', center=0, ax=axes[i],
                cbar_kws={'label': 'Loading'})
    axes[i].set_title(f'{pc} loadings (important features)')
plt.tight_layout()
stamp_version(fig, label='02 important features')
plt.savefig(EXAMPLE_OUTPUT_DIR / '02_loadings_important_features.png', dpi=150, bbox_inches='tight')
plt.show()


**What this figure shows.** Same cross-phase heatmap as figure 4,
but restricted to only the important-feature union we just
identified. Cleaner view of what's actually doing variance work
once we drop the noise-floor features.

**What to read into it.** Same patterns as figure 4 -- stable rows
are phase-invariant kinematic axes; flipping rows are
phase-dependent. The point of this filtered version is that every
row here is "important enough to track"; the rows that didn't
survive the union are gone, so any pattern you see now is on a
feature worth interpreting.
